In [ ]:
import os
import numpy as np
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import json
%matplotlib inline

from tensorboard.backend.event_processing import event_accumulator
from collections import defaultdict


In [ ]:
# list experiments
NUM_EXPERIMENTS = 2
root = "/mnt/storage/log-experiments/"
log_root = root + "logs/"

# only want last 15 experiments
# experiments = [f for f in os.listdir(log_root) if f[-4:] == 'json']

# place them in a dictionary
exp_dict = {}
for exp in os.listdir(log_root)[-NUM_EXPERIMENTS:]:
    if exp[-4:] == 'json':
        exp_id = exp[:-5]
        with open(log_root+exp, "rb") as f:
            exp_dict[exp_id] = json.load(f)
            
# convert dictionary to a dataframe
experiments_df = pd.DataFrame.from_dict(exp_dict, orient='index')

In [ ]:
experiments_df.shape

In [ ]:
experiments_df.index

In [ ]:
experiments_df.head()

In [ ]:
runs_root = root + "runs/"
experiments_runs = os.listdir(runs_root)[-NUM_EXPERIMENTS:]
experiments_runs

In [ ]:
runs_root = root + "runs/"
experiments_data = {}
for experiment in experiments_df.index:
    if experiment != ".DS_Store":
        for trial in os.listdir(os.path.join(runs_root, experiment)):
            if trial != ".DS_Store":
                for run in os.listdir(os.path.join(runs_root, experiment, trial)):
                    if trial != ".DS_Store":                
                        ea = event_accumulator.EventAccumulator(os.path.join(runs_root, experiment, trial, run))
                        ea.Reload()
                        eid = "-".join([experiment, trial])
                        experiments_data[eid] = None
                        for scalar in ea.Tags()['scalars']:
                            df_temp = pd.DataFrame(ea.Scalars(scalar)).drop('wall_time', axis=1).rename(index=str, columns={'value': scalar})
                            if experiments_data[eid] is None:
                                experiments_data[eid] = df_temp
                            else:
                                experiments_data[eid] = pd.merge(experiments_data[eid], df_temp, how='outer', on='step')
                                

In [ ]:
# listing all experiments
experiments_data.keys()

In [ ]:
len(experiments_data.keys())

In [ ]:
# getting columns from all experiments
sample = list(experiments_data.keys())[1]
experiments_data[sample].columns

In [ ]:
# example comparing rewards from within an experiment
plt.figure(figsize=(12,6))
exp_name = sample
for exp,data in experiments_data.items():
    if exp.startswith(exp_name):
        plt.plot(data['reward_100/avg'], label=exp_name)


In [ ]:
# they don't all fit the same timeframe
rewards_exp = defaultdict(list)
for key in experiments_data.keys():
    exp_name = key[:len(experiments_df.index[0])]
    num_episodes = len(experiments_data[key][['reward']].dropna())
    rewards_exp[exp_name].append(num_episodes)

experiment_results = list(zip(rewards_exp.keys(), map(lambda x:np.mean(x), rewards_exp.values())))

df_new = pd.DataFrame(experiment_results, columns=['exp', 'episodes'])
df_new = df_new.set_index('exp')


In [ ]:
df_new

In [ ]:
experiments_df.head()

In [ ]:
experiments_df_rev = pd.merge(experiments_df, df_new, left_index=True, right_index=True)

In [ ]:
experiments_df_rev.sort_values(by='episodes', ascending=True)

In [ ]:
experiments_df_rev[['GAMMA', 'BATCH_SIZE', 'REPLAY_SIZE', 'LEARNING_RATE', 'SOFT_UPDATE_TAU', 'EPSILON_DECAY_LAST_FRAME', 'SHARING', 'episodes']]

In [ ]:
# they don't all fit the same timeframe
steps_exp = defaultdict(list)
for key in experiments_data.keys():
    exp_name = key[:len(experiments_df.index[0])]
    num_episodes = experiments_data[key][['steps']].dropna()
    steps_exp[exp_name].append(num_episodes.iloc[-1])
    
experiment_results = list(zip(steps_exp.keys(), map(lambda x:np.mean(x), steps_exp.values())))

df_new = pd.DataFrame(experiment_results, columns=['exp', 'mean of steps'])
df_new = df_new.set_index('exp')
df_new



In [ ]:
# they don't all fit the same timeframe
aggregate = defaultdict(list)
for key in experiments_data.keys():
    exp_name = key[:len(experiments_df.index[0])]
    num_episodes = experiments_data[key][['reward']].dropna()
    aggregate[exp_name].append(num_episodes.iloc[-1])

    experiment_results = list(zip(steps_exp.keys(), map(lambda x:np.mean(x), aggregate.values())))

df_new = pd.DataFrame(experiment_results, columns=['exp', 'mean of rewards'])
df_new = df_new.set_index('exp')
df_new


Mean of steps is a a lot lower, but there is no significant difference in mean of rewards. It would be interesting to see the distribution of steps, rewards and number of episodes as well.

There is a significant difference however in number of episodes (50 against 62), which is the problem that matter. What happened is that the experience sharing compensated for the lack of exploration. However, I still don't see significant improvement over the configuration of epsilon_decay_last_frame=3000, with a total of 50.78 episodes against 51.81 or 52.47. Hence, there is no real gain that could not be achieved with a modification of the original hyperparameters (namely the exploration rate). But it could be useful in robot scenarios where exploration is costly, since it significantly reduces the number of steps.

Those results are good enough to include in the publication. But ideally I want to test more scenarios, like the prioritized version as well. The paper can be done as a compilation of small improvements. 


In [ ]:
# they don't all fit the same timeframe
count_episodes1 = defaultdict(list)
for key in experiments_data.keys():
    exp_name = key[:len(experiments_df.index[0])]
    num_episodes = len(experiments_data[key][['reward']].dropna())
    count_episodes1[exp_name].append(num_episodes)



In [ ]:
count_episodes2 = defaultdict(list)
for key in experiments_data.keys():
    exp_name = key[:len(experiments_df.index[0])]
    num_episodes = len(experiments_data[key][['steps']].dropna())
    count_episodes2[exp_name].append(num_episodes)



In [ ]:
count_episodes1

In [ ]:
for k,v in count_episodes1.items():
    for i in v:
        if i < 10:
            print(k, i)

Results are not to be celebrated. They can be a simple issue with the way I'm reporting. There shouldn't be any trial with 1 episode, however, there are 7 (and 6 of them in the first trial). Open up local log and compare result to see if they are consistent.

In [ ]:
new_dict = {}
for k,v in count_episodes1.items():
    new_dict[k] = []
    for i in v:
        if i > 10:
            new_dict[k].append(i)

In [ ]:
for k,v in new_dict.items():
    print(np.mean(v))

Results are still better removing outliers, but there are far too many outliers. What if their results are significantly higher?